# SSL400 Sinhala Sign Language — Google Colab Training
## Run each cell ONE AT A TIME and wait for it to finish!
---
| EXP | Enhancement | MixUp | Augmentation |
|-----|-------------|-------|--------------|
| 1   | None (Baseline) | OFF | OFF |
| 2   | CLAHE + Gamma   | ON  | ON  |
| 3   | Bilateral + Unsharp | ON | ON |
| 4   | Hybrid (All)    | ON  | ON  |

**Google Drive folder structure required:**
```
MyDrive/SSL400_Research/
    data/
        raw/          ← Upload your 8 sign class folders here!
        splits/       ← Will be copied from the zip automatically
    src/              ← Will be copied from the zip automatically
    config.yaml       ← Will be copied from the zip automatically
```

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES & MOUNT GOOGLE DRIVE
# ============================================================
!pip install tf-keras ultralytics --quiet

from google.colab import drive
drive.mount('/content/drive')

print('Drive mounted! Dependencies installed!')

In [ ]:
# ============================================================
# CELL 2: SET UP WORKING DIRECTORY FROM GOOGLE DRIVE
# ============================================================
import os, shutil

# Your Google Drive project folder
DRIVE_DIR = '/content/drive/MyDrive/SSL400_Research'

# Runtime working directory (fast SSD)
WORK_DIR = '/content/working'

# Create working directories
os.makedirs(f'{WORK_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data/splits', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/logs', exist_ok=True)

# Copy code from Drive to runtime (fast access)
os.system(f'cp -r {DRIVE_DIR}/src {WORK_DIR}/')
os.system(f'cp -r {DRIVE_DIR}/data/splits {WORK_DIR}/data/')
os.system(f'cp {DRIVE_DIR}/config.yaml {WORK_DIR}/')

# Link raw videos (keep on Drive to save runtime disk space)
raw_src = f'{DRIVE_DIR}/data/raw'
raw_dst = f'{WORK_DIR}/data/raw'
if not os.path.exists(raw_dst):
    os.symlink(raw_src, raw_dst)

# Verify
print('Split files:', os.listdir(f'{WORK_DIR}/data/splits'))
print('Raw classes:', os.listdir(f'{WORK_DIR}/data/raw'))
print('✅ Setup complete!')

In [ ]:
# ============================================================
# CELL 3: APPLY ALL BUG FIXES (Run ONCE per session)
# ============================================================

# --- FIX 1: EfficientNetV2 Normalization Fix ---
file_path = f'{WORK_DIR}/src/data/tf_dataset_builder.py'
with open(file_path, 'r') as f:
    code = f.read()

if '_unnormalize' not in code:
    fix = '''
    # CRITICAL FIX: Un-normalize for EfficientNetV2 [0,255]
    def _unnormalize(frames, labels):
        frames = (frames + 1.0) * 127.5
        return frames, labels
    
    ds = ds.map(_unnormalize, num_parallel_calls=AUTOTUNE)

    ds = ds.prefetch(AUTOTUNE)

    return ds
'''
    code = code.replace('    ds = ds.prefetch(AUTOTUNE)\n\n    return ds', fix)
    with open(file_path, 'w') as f:
        f.write(code)
    print('FIX 1: Normalization patch applied!')
else:
    print('FIX 1: Already applied, skipping.')

# --- FIX 2: MixUp OFF for EXP1, ON for EXP2-4 ---
train_path = f'{WORK_DIR}/src/training/train.py'
with open(train_path, 'r') as f:
    train_code = f.read()

if 'exp.get("use_augmentation"' not in train_code:
    train_code = train_code.replace(
        'use_mixup=(p1["mixup_alpha"] > 0),',
        'use_mixup=(p1["mixup_alpha"] > 0 and exp.get("use_augmentation", True)),'
    )
    train_code = train_code.replace(
        'use_mixup=(p2["mixup_alpha"] > 0),',
        'use_mixup=(p2["mixup_alpha"] > 0 and exp.get("use_augmentation", True)),'
    )
    with open(train_path, 'w') as f:
        f.write(train_code)
    print('FIX 2: MixUp patch applied! (OFF for EXP1, ON for EXP2-4)')
else:
    print('FIX 2: Already applied, skipping.')

print('\nAll fixes applied! Ready to train.')

In [ ]:
# ============================================================
# CELL 4: SET EXPERIMENT ID
#   EXP_ID = 1  Baseline (No Enhancement, No MixUp)
#   EXP_ID = 2  CLAHE + Gamma + MixUp
#   EXP_ID = 3  Bilateral + Unsharp + MixUp
#   EXP_ID = 4  Hybrid + MixUp
# ============================================================
EXP_ID = 1

import yaml, os, sys
os.chdir(WORK_DIR)
sys.path.insert(0, f'{WORK_DIR}/src')

with open(f'{WORK_DIR}/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
exp = config['experiments'][EXP_ID - 1]
print(f'\n{"="*50}')
print(f'  Running: EXP {EXP_ID} — {exp["name"]}')
print(f'  Augmentation: {exp["use_augmentation"]}')
print(f'  MixUp: {"ON" if exp["use_augmentation"] else "OFF"}')
print(f'  Resolution: {config["frames"]["width"]}x{config["frames"]["height"]}')
print(f'{"="*50}\n')

In [ ]:
# ============================================================
# RESUME CELL — Run after ANY session disconnect, before Cell 7!
# ============================================================
import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/SSL400_Research'
WORK_DIR  = '/content/working'

model_dst = f'{WORK_DIR}/models/experiment_{EXP_ID}'
model_src = f'{DRIVE_DIR}/models/experiment_{EXP_ID}'
os.makedirs(model_dst, exist_ok=True)

for fname in ['best_model_phase1.keras', 'best_model_phase2.keras']:
    src = f'{model_src}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{model_dst}/{fname}')
        print(f'Restored: {fname}')
    else:
        print(f'Not found (OK if not yet saved): {fname}')

log_dst = f'{WORK_DIR}/logs/experiment_{EXP_ID}'
log_src = f'{DRIVE_DIR}/logs/experiment_{EXP_ID}'
os.makedirs(log_dst, exist_ok=True)

for fname in ['training_log_phase1.csv', 'training_log_phase2.csv']:
    src = f'{log_src}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{log_dst}/{fname}')
        print(f'Restored: {fname}')
    else:
        print(f'Not found (OK if Phase 1 not done yet): {fname}')

print('\n✅ Resume complete! Now run Cell 7 to continue!')


In [ ]:
# ============================================================
# CELL 5: PROCESS VIDEOS → ENHANCED FRAMES
# ============================================================
!cd {WORK_DIR} && python src/data/video_to_frames.py --exp_id {EXP_ID}

In [ ]:
# ============================================================
# CELL 6: VERIFY DATA IS CORRECTLY LOADED
# ============================================================
import pandas as pd
from pathlib import Path

proc_dir = Path(WORK_DIR) / exp['processed_dir']

for split in ['train', 'val', 'test']:
    csv_path = f'{WORK_DIR}/data/splits/{split}_split.csv'
    df = pd.read_csv(csv_path)
    found = sum(
        (proc_dir / str(int(row['class_id'])) / f"{Path(str(row['video_path'])).stem}.npy").exists()
        for _, row in df.iterrows()
    )
    status = 'OK' if found == len(df) else 'MISSING FILES'
    print(f'{status} {split.upper()}: {found}/{len(df)} files found')

print('\nIf all show OK, you are ready to train!')

In [ ]:
# ============================================================
# CELL 7: TRAIN (PHASE 1 + PHASE 2)
# Training auto-saves to Google Drive for session safety!
# ============================================================
!cd {WORK_DIR} && python src/training/train.py --exp_id {EXP_ID} --drive_dir {DRIVE_DIR}/models/experiment_{EXP_ID}

In [ ]:
# ============================================================
# CELL 8: EVALUATE — CLASSIFICATION REPORT
# ============================================================
import os, sys, yaml, numpy as np
import tensorflow as tf
import tf_keras as keras
os.chdir(WORK_DIR)
sys.path.insert(0, f'{WORK_DIR}/src')

from models.efficientnet_builder import build_model
from data.tf_dataset_builder import build_dataset
from sklearn.metrics import classification_report

with open(f'{WORK_DIR}/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

H = config['frames']['height']
W = config['frames']['width']
N = config['frames']['num_frames']
exp = config['experiments'][EXP_ID - 1]

test_ds = build_dataset(
    split_csv=f'{WORK_DIR}/data/splits/test_split.csv',
    processed_dir=f"{WORK_DIR}/{exp['processed_dir']}",
    num_classes=8, batch_size=2, is_training=False,
    num_frames=N, target_size=(W, H)
)

model_path = f'{WORK_DIR}/models/experiment_{EXP_ID}/best_model_phase2.keras'
if os.path.exists(model_path):
    model = build_model(num_classes=8, num_frames=N,
                        img_height=H, img_width=W,
                        lstm_units=512, dropout_rate=0.4)
    model.load_weights(model_path)
    true_labels, predictions = [], []
    for frames, labels in test_ds:
        preds = model.predict(frames, verbose=0)
        true_labels.extend(np.argmax(labels.numpy(), axis=1))
        predictions.extend(np.argmax(preds, axis=1))
    label_map = {0:'Thank you', 1:'Hello', 2:'Good', 3:'House',
                 4:'Eat', 5:'Drink', 6:'Tell', 7:'Write'}
    print(f"\n{'='*50}\n  EXP {EXP_ID}: {exp['name']}\n{'='*50}")
    print(classification_report(true_labels, predictions,
          target_names=[label_map[i] for i in range(8)], zero_division=0))
else:
    print(f'Model not found at {model_path}. Did training complete?')

In [ ]:
# ============================================================
# CELL 9: GENERATE THESIS PLOTS
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

label_map = {0:'Thank you', 1:'Hello', 2:'Good', 3:'House',
             4:'Eat', 5:'Drink', 6:'Tell', 7:'Write'}
class_names = [label_map[i] for i in range(8)]
exp_name = config['experiments'][EXP_ID-1]['name']

# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'EXP {EXP_ID}: {exp_name}\nConfusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
out_path = f'{WORK_DIR}/exp{EXP_ID}_confusion_matrix.png'
plt.savefig(out_path, dpi=150); plt.show(); plt.close()

# Training Curves
try:
    p1 = pd.read_csv(f'{WORK_DIR}/logs/experiment_{EXP_ID}/training_log_phase1.csv')
    p2 = pd.read_csv(f'{WORK_DIR}/logs/experiment_{EXP_ID}/training_log_phase2.csv')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    offset = len(p1)
    axes[0].plot(p1['accuracy'], 'b--', label='P1 Train')
    axes[0].plot(p1['val_accuracy'], 'b-', label='P1 Val')
    axes[0].plot(range(offset, offset+len(p2)), p2['accuracy'], 'r--', label='P2 Train')
    axes[0].plot(range(offset, offset+len(p2)), p2['val_accuracy'], 'r-', label='P2 Val')
    axes[0].set_title(f'EXP {EXP_ID} Accuracy', fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(p1['loss'], 'b--', label='P1 Train')
    axes[1].plot(p1['val_loss'], 'b-', label='P1 Val')
    axes[1].plot(range(offset, offset+len(p2)), p2['loss'], 'r--', label='P2 Train')
    axes[1].set_title(f'EXP {EXP_ID} Loss', fontweight='bold')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{WORK_DIR}/exp{EXP_ID}_training_curves.png', dpi=150)
    plt.show(); plt.close()
    print(f'EXP {EXP_ID} plots saved!')
except Exception as e:
    print(f'Could not generate training curves: {e}')

In [ ]:
# ============================================================
# CELL 10: SAVE RESULTS TO GOOGLE DRIVE
# ============================================================
import os, shutil

drive_results = f'{DRIVE_DIR}/results/experiment_{EXP_ID}'
os.makedirs(drive_results, exist_ok=True)

# Copy plots
for fname in [f'exp{EXP_ID}_confusion_matrix.png', f'exp{EXP_ID}_training_curves.png']:
    src = f'{WORK_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{drive_results}/{fname}')

# Copy logs
log_src = f'{WORK_DIR}/logs/experiment_{EXP_ID}'
log_dst = f'{DRIVE_DIR}/logs/experiment_{EXP_ID}'
if os.path.exists(log_src):
    shutil.copytree(log_src, log_dst, dirs_exist_ok=True)

# Copy best model
model_src = f'{WORK_DIR}/models/experiment_{EXP_ID}/best_model_phase2.keras'
model_dst = f'{DRIVE_DIR}/models/experiment_{EXP_ID}/'
os.makedirs(model_dst, exist_ok=True)
if os.path.exists(model_src):
    shutil.copy2(model_src, model_dst)

print(f'EXP {EXP_ID} results saved to Google Drive!')
print(f'Location: {drive_results}')

In [ ]:
# ============================================================
# CELL 11: CLEANUP PROCESSED FRAMES (Run before next experiment!)
# Frees up runtime disk space. Models and logs are safe in Drive!
# ============================================================
import shutil, yaml
with open(f'{WORK_DIR}/config.yaml') as f:
    config = yaml.safe_load(f)
exp = config['experiments'][EXP_ID - 1]
proc_dir = f"{WORK_DIR}/{exp['processed_dir']}"
shutil.rmtree(proc_dir, ignore_errors=True)
print(f'Deleted: {proc_dir}')
print('Now change EXP_ID in Cell 4 and run from Cell 5!')

In [ ]:
# ============================================================
# SAVE CLASSIFICATION REPORT TO GOOGLE DRIVE
# Run this AFTER Cell 8 to save the text report
# ============================================================
import os
from sklearn.metrics import classification_report

report_text = classification_report(
    true_labels, 
    predictions, 
    target_names=[label_map[i] for i in range(8)], 
    zero_division=0
)

res_dir = f'/content/drive/MyDrive/SSL400_Research/results/experiment_{EXP_ID}'
os.makedirs(res_dir, exist_ok=True)

report_path = f'{res_dir}/exp{EXP_ID}_classification_report.txt'
with open(report_path, 'w') as f:
    f.write(f"====================================================\n")
    f.write(f"  EXP {EXP_ID} Classification Report\n")
    f.write(f"====================================================\n\n")
    f.write(report_text)

print(f"✅ Full Classification Report saved to: {report_path}")
